# Week 12: Deployment Validation

This notebook validates the deployed API through its SSH tunnel before public release. It reuses the frozen Week 11 evaluation set to compare search quality, cache-aware latency, detail retrieval, and the bounded quality-rerank path.

---

In [1]:
import json
import math
import os
import time
import uuid
from collections import defaultdict
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_colwidth", 180)

API_BASE_URL = os.getenv("API_BASE_URL", "http://127.0.0.1:8000").rstrip("/")
TOP_K = 5
PROFILES = ["fast", "balanced", "quality"]
PROFILE_ORDER = pd.CategoricalDtype(categories=PROFILES, ordered=True)
REQUEST_PAUSE_SECONDS = 0.9
REQUEST_TIMEOUT_SECONDS = 60

REQUEST_HEADERS = {"X-Search-Session-ID": uuid.uuid4().hex}

session = requests.Session()
session.headers.update(REQUEST_HEADERS)

## 1. Deployment Artifacts and API Readiness

Start an SSH tunnel before running this notebook: `ssh -L 8000:127.0.0.1:8000 <vm>`. The relevance labels were frozen against the pre-web snapshot. This section verifies that the deployed API still serves the same retrieval population and model settings before reporting benchmark results.

---

In [2]:
query_dataset = json.loads(Path("../data/processed/search_relevance_queries.json").read_text())
final_manifest = json.loads(Path("../data/processed/search_relevance_final_manifest.json").read_text())
qrels = [
    json.loads(line)
    for line in Path("../data/processed/search_relevance_final_qrels.jsonl").read_text().splitlines()
    if line
]

query_items = query_dataset["items"]
test_items = [item for item in query_items if item["split"] == "test"]

ready_response = session.get(f"{API_BASE_URL}/ready", timeout=REQUEST_TIMEOUT_SECONDS)
ready_response.raise_for_status()
ready = ready_response.json()

active_pointer = json.loads(Path("../data/models/search/active.json").read_text())
active_snapshot_dir = Path(active_pointer["snapshot_path"])
active_manifest = json.loads((active_snapshot_dir / "manifest.json").read_text())
frozen_snapshot_dir = Path("../data/models/search_snapshots") / final_manifest["snapshot_id"]
frozen_snapshot_manifest = json.loads((frozen_snapshot_dir / "manifest.json").read_text())

compatibility_fields = [
    "public_listing_count",
    "retrievable_listing_count",
    "compliance_rule_version",
    "dense_model",
]
compatibility = pd.DataFrame([
    {
        "field": field,
        "frozen benchmark": frozen_snapshot_manifest[field],
        "active API": active_manifest[field],
        "matches": frozen_snapshot_manifest[field] == active_manifest[field],
    }
    for field in compatibility_fields
])

assert ready["snapshot_id"] == active_manifest["snapshot_id"]
assert compatibility["matches"].all()

display(pd.DataFrame([
    {
        "api_base_url": API_BASE_URL,
        "active_snapshot": ready["snapshot_id"],
        "frozen_benchmark_snapshot": final_manifest["snapshot_id"],
        "benchmark_queries": len(query_items),
        "held_out_test_queries": len(test_items),
        "qrels": len(qrels),
    }
]))
compatibility

,api_base_url,active_snapshot,frozen_benchmark_snapshot,benchmark_queries,held_out_test_queries,qrels
0,http://127.0.0.1:8012,20260905T052007Z_5466f5d6de,20260903T124929Z_5466f5d6de,40,12,849


,field,frozen benchmark,active API,matches
0,public_listing_count,53091,53091,True
1,retrievable_listing_count,52763,52763,True
2,compliance_rule_version,federal-1.1,federal-1.1,True
3,dense_model,sentence-transformers/all-MiniLM-L6-v2,sentence-transformers/all-MiniLM-L6-v2,True


## 2. Live API Benchmark

Each query/profile pair is sent once, then repeated immediately with the same payload. This produces an actual cache miss and cache hit for every pair. The request pace stays below the production rate limit, and the notebook uses a separate session ID so it does not share a rate-limit bucket with the browser.

---

In [3]:
def post_json(path, payload):
    for attempt in range(2):
        started_at = time.perf_counter()
        response = session.post(
            f"{API_BASE_URL}{path}",
            json=payload,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        elapsed_ms = (time.perf_counter() - started_at) * 1000
        if response.status_code == 429 and attempt == 0:
            time.sleep(float(response.headers.get("Retry-After", 1)))
            continue
        response.raise_for_status()
        return response.json(), elapsed_ms, response.headers
    raise RuntimeError("Request remained rate limited after one retry.")


def search_request(item, profile):
    body, elapsed_ms, headers = post_json(
        "/search",
        {
            "query": item["query"],
            "top_k": TOP_K,
            "sort_by": None,
            "search_profile": profile,
        },
    )
    return body, {
        "query_id": item["id"],
        "split": item["split"],
        "category": item["category"],
        "profile": profile,
        "http_status": 200,
        "cache_status": headers.get("X-Cache", "BYPASS"),
        "observed_latency_ms": round(elapsed_ms, 2),
        "reported_compute_latency_ms": body["meta"]["timings_ms"].get("total"),
        "cross_encoder_rerank_ms": body["meta"]["timings_ms"].get("cross_encoder_rerank"),
        "effective_profile": body["meta"]["effective_profile"],
        "reranker_used": body["meta"]["reranker_used"],
        "returned": len(body["results"]),
        "result_ids": [str(row["listing_id"]) for row in body["results"]],
    }


response_by_key = {}
benchmark_rows = []

for profile in PROFILES:
    for item in query_items:
        for phase in ["initial", "repeat"]:
            body, row = search_request(item, profile)
            row["phase"] = phase
            benchmark_rows.append(row)
            response_by_key[phase, profile, item["id"]] = body
            time.sleep(REQUEST_PAUSE_SECONDS)

benchmark = pd.DataFrame(benchmark_rows)
benchmark["profile"] = benchmark["profile"].astype(PROFILE_ORDER)
assert (benchmark["http_status"] == 200).all()
assert (benchmark["returned"] == TOP_K).all()

execution_summary = benchmark.groupby(
    ["phase", "profile", "cache_status"],
    as_index=False,
    observed=True,
).agg(
    requests=("query_id", "count"),
    returned_results=("returned", "sum"),
)
execution_summary

,phase,profile,cache_status,requests,returned_results
0,initial,fast,MISS,40,200
1,initial,balanced,MISS,40,200
2,initial,quality,MISS,40,200
3,repeat,fast,HIT,40,200
4,repeat,balanced,HIT,40,200
5,repeat,quality,HIT,40,200


## 3. Held-Out Search Quality

The final quality score uses only the 12 held-out queries. A result is relevant at grade 2 or above; NDCG keeps the full four-level relevance scale. Judgment coverage is checked before any score is reported.

---

In [4]:
RELEVANT_GRADE = 2
qrel_by_key = {
    (row["query_id"], str(row["listing_id"])): row["relevance_grade"]
    for row in qrels
}
grades_by_query = defaultdict(list)
for row in qrels:
    grades_by_query[row["query_id"]].append(row["relevance_grade"])


def precision_at_five(grades):
    return sum(grade >= RELEVANT_GRADE for grade in grades) / TOP_K


def mrr_at_five(grades):
    for rank, grade in enumerate(grades, start=1):
        if grade >= RELEVANT_GRADE:
            return 1 / rank
    return 0.0


def ndcg_at_five(grades, ideal_grades):
    def dcg(values):
        return sum((2 ** grade - 1) / math.log2(rank + 1) for rank, grade in enumerate(values, start=1))

    ideal = dcg(sorted(ideal_grades, reverse=True)[:TOP_K])
    return dcg(grades) / ideal if ideal else 0.0


quality_rows = []
for profile in PROFILES:
    for item in test_items:
        result = response_by_key["initial", profile, item["id"]]
        result_ids = [str(row["listing_id"]) for row in result["results"]]
        missing = [listing_id for listing_id in result_ids if (item["id"], listing_id) not in qrel_by_key]
        quality_rows.append({
            "query_id": item["id"],
            "profile": profile,
            "judged_top_5": TOP_K - len(missing),
            "missing_judgments": missing,
            "grades": [qrel_by_key.get((item["id"], listing_id)) for listing_id in result_ids],
        })

quality_by_query = pd.DataFrame(quality_rows)
quality_by_query["profile"] = quality_by_query["profile"].astype(PROFILE_ORDER)
assert (quality_by_query["judged_top_5"] == TOP_K).all()

quality_by_query["precision_at_5"] = quality_by_query["grades"].apply(precision_at_five)
quality_by_query["mrr_at_5"] = quality_by_query["grades"].apply(mrr_at_five)
quality_by_query["ndcg_at_5"] = quality_by_query.apply(
    lambda row: ndcg_at_five(row["grades"], grades_by_query[row["query_id"]]),
    axis=1,
)

quality_summary = quality_by_query.groupby("profile", as_index=False, observed=True).agg(
    held_out_queries=("query_id", "count"),
    judged_top_5_coverage=("judged_top_5", lambda values: values.sum() / (len(values) * TOP_K)),
    precision_at_5=("precision_at_5", "mean"),
    ndcg_at_5=("ndcg_at_5", "mean"),
    mrr_at_5=("mrr_at_5", "mean"),
)
quality_summary.round(3)

,profile,held_out_queries,judged_top_5_coverage,precision_at_5,ndcg_at_5,mrr_at_5
0,fast,12,1.0,0.667,0.566,0.794
1,balanced,12,1.0,0.800,0.744,0.917
2,quality,12,1.0,0.867,0.901,0.917


### 3.1 Hard-Filter Integrity

Relevance should not come at the cost of explicit requirements. The held-out set contains city, maximum-price, and minimum-bedroom constraints, which can be checked directly against the public result fields.


In [5]:
def meets_hard_requirements(result, requirements):
    if "city" in requirements and str(result.get("city", "")).casefold() != requirements["city"].casefold():
        return False
    if "price_max" in requirements and (result.get("price") is None or result["price"] > requirements["price_max"]):
        return False
    if "beds_min" in requirements and (result.get("beds") is None or result["beds"] < requirements["beds_min"]):
        return False
    return True


filter_rows = []
for profile in PROFILES:
    for item in test_items:
        requirements = item["hard_requirements"]
        result = response_by_key["initial", profile, item["id"]]
        matches = [meets_hard_requirements(row, requirements) for row in result["results"]]
        filter_rows.append({
            "query_id": item["id"],
            "profile": profile,
            "has_explicit_filter": bool(requirements),
            "valid_results": sum(matches),
            "returned_results": len(matches),
            "query_fully_valid": all(matches),
        })

filter_integrity = pd.DataFrame(filter_rows)
filter_integrity["profile"] = filter_integrity["profile"].astype(PROFILE_ORDER)
filter_integrity.groupby("profile", as_index=False, observed=True).agg(
    held_out_queries=("query_id", "count"),
    queries_with_explicit_filters=("has_explicit_filter", "sum"),
    result_constraint_accuracy=("valid_results", "sum"),
    total_results=("returned_results", "sum"),
    fully_valid_queries=("query_fully_valid", "sum"),
).assign(
    result_constraint_accuracy=lambda frame: frame["result_constraint_accuracy"] / frame["total_results"],
    fully_valid_query_rate=lambda frame: frame["fully_valid_queries"] / frame["held_out_queries"],
).round(3)

,profile,held_out_queries,queries_with_explicit_filters,result_constraint_accuracy,total_results,fully_valid_queries,fully_valid_query_rate
0,fast,12,8,1.0,60,12,1.0
1,balanced,12,8,1.0,60,12,1.0
2,quality,12,8,1.0,60,12,1.0


## 4. Cache-Aware Latency

Observed latency is the client-visible HTTP time. API compute latency and Cross Encoder timing are interpreted only for cache misses because cached responses retain the original search metadata. The tables group by the actual cache header instead of assuming the initial pass was cold.

---

In [6]:
def latency_summary(frame, column):
    values = frame[column].dropna()
    return pd.Series({
        "requests": len(values),
        "p50_ms": values.quantile(0.50),
        "p90_ms": values.quantile(0.90),
        "p95_ms": values.quantile(0.95),
    })


http_latency = (
    benchmark.groupby(
        ["profile", "phase", "cache_status"],
        group_keys=False,
        observed=True,
    )
    .apply(lambda frame: latency_summary(frame, "observed_latency_ms"))
    .reset_index()
    .sort_values(["profile", "phase", "cache_status"])
)

api_compute_latency = (
    benchmark.query("phase == 'initial' and cache_status == 'MISS'")
    .groupby("profile", group_keys=False, observed=True)
    .apply(lambda frame: latency_summary(frame, "reported_compute_latency_ms"))
    .reset_index()
    .sort_values("profile")
)

display(http_latency.round(2))
display(api_compute_latency.round(2))

rerank_latency = (
    benchmark.query("profile == 'quality' and phase == 'initial' and cache_status == 'MISS'")
    .pipe(lambda frame: latency_summary(frame, "cross_encoder_rerank_ms"))
    .to_frame().T
)
rerank_latency.round(2)

,profile,phase,cache_status,requests,p50_ms,p90_ms,p95_ms
0,fast,initial,MISS,40.0,162.20,547.76,552.90
1,fast,repeat,HIT,40.0,67.04,73.86,76.63
2,balanced,initial,MISS,40.0,172.68,679.52,728.80
3,balanced,repeat,HIT,40.0,69.75,76.51,78.25
4,quality,initial,MISS,40.0,4133.26,4578.21,4652.66
5,quality,repeat,HIT,40.0,69.52,75.07,76.73


,profile,requests,p50_ms,p90_ms,p95_ms
0,fast,40.0,81.16,425.90,490.07
1,balanced,40.0,106.13,609.55,678.31
2,quality,40.0,4081.60,4502.86,4583.14


,requests,p50_ms,p90_ms,p95_ms
0,40.0,3823.23,4035.52,4183.96


## 5. Listing Details Flow

The product page loads details in batches after search results are available. This checks the same `/listings/details` path for every held-out Quality result page and measures its miss/hit behavior.

---

In [7]:
detail_rows = []
details_by_query = {}

for phase in ["initial", "repeat"]:
    for item in test_items:
        search_result = response_by_key["initial", "quality", item["id"]]
        listing_ids = [str(row["listing_id"]) for row in search_result["results"]]
        body, elapsed_ms, headers = post_json("/listings/details", {"listing_ids": listing_ids})
        details = body["listings"]
        detail_rows.append({
            "query_id": item["id"],
            "phase": phase,
            "cache_status": headers.get("X-Cache", "BYPASS"),
            "observed_latency_ms": round(elapsed_ms, 2),
            "requested": len(listing_ids),
            "returned": len(details),
            "ids_match": [str(row["listing_id"]) for row in details] == listing_ids,
            "descriptions_present": sum(bool(row.get("listing_description")) for row in details),
        })
        details_by_query[phase, item["id"]] = {str(row["listing_id"]): row for row in details}
        time.sleep(REQUEST_PAUSE_SECONDS)

detail_benchmark = pd.DataFrame(detail_rows)
assert detail_benchmark["ids_match"].all()
assert (detail_benchmark["descriptions_present"] == TOP_K).all()

detail_integrity = detail_benchmark.groupby("phase", as_index=False).agg(
    batches=("query_id", "count"),
    cache_states=("cache_status", lambda values: ", ".join(sorted(values.unique()))),
    requested_listings=("requested", "sum"),
    returned_listings=("returned", "sum"),
    descriptions_present=("descriptions_present", "sum"),
)

detail_latency = (
    detail_benchmark.groupby(["phase", "cache_status"], group_keys=False)
    .apply(lambda frame: latency_summary(frame, "observed_latency_ms"))
    .reset_index()
)

display(detail_integrity)
detail_latency.round(2)

,phase,batches,cache_states,requested_listings,returned_listings,descriptions_present
0,initial,12,MISS,60,60,60
1,repeat,12,HIT,60,60,60


,phase,cache_status,requests,p50_ms,p90_ms,p95_ms
0,initial,MISS,12.0,71.88,75.66,77.43
1,repeat,HIT,12.0,68.53,75.53,75.82


## 6. Product Examples

These examples use the default `quality` profile and show the response fields that feed its search cards: structured facts, summaries, matched preferences, and the original listing description returned by the batch-details endpoint.

---

In [8]:
example_ids = ["rel_029", "rel_032", "rel_039"]
example_rows = []

for query_id in example_ids:
    item = next(item for item in test_items if item["id"] == query_id)
    result = response_by_key["initial", "quality", query_id]
    details = details_by_query["initial", query_id]
    for row in result["results"][:3]:
        detail = details[str(row["listing_id"])]
        example_rows.append({
            "query": item["query"],
            "rank": row["rank"],
            "address": row["address"],
            "city": row["city"],
            "price": row["price"],
            "beds": row["beds"],
            "baths": row["baths"],
            "sqft": row["sqft"],
            "matched_preferences": ", ".join(match["value"] for match in row["matched_signals"]),
            "summary": row["summary"],
            "description_excerpt": detail["listing_description"][:260],
        })

pd.DataFrame(example_rows)

,query,rank,address,city,price,beds,baths,sqft,matched_preferences,summary,description_excerpt
0,Homes in Irvine under $1m with a private pool,1,45 Alicante Aisle 52,Irvine,849000,2,2.0,1032,,"This 2-bed, 2-bath listing in Irvine is listed at $849,000. Highlights include a pool and beach access.","Rivaling brand-new model homes, this move-in-ready Irvine condo in the village of Westpark offers a\r\r\nsought-after end location opening to a broad lawn with mature trees and..."
1,Homes in Irvine under $1m with a private pool,2,2408 Watermarke Place,Irvine,899000,2,2.0,1260,,"This 2-bed, 2-bath listing in Irvine is listed at $899,000. Highlights include a pool and an open floor plan.","Luxury Top-Floor Penthouse Life Style with Panoramic Pool Views at Watermarke, Irvine.\r\r\n\r\r\nWelcome! This top-floor 2-bedroom, 2-bath condo offers total privacy with no ..."
2,Homes in Irvine under $1m with a private pool,3,113 Tarocco,Irvine,799900,2,2.0,951,,"This 2-bed, 2-bath listing in Irvine is listed at $799,900. Highlights include a pool and a remodeled interior.","Welcome to 113 Tarocco in the heart of Irvine! This beautifully remodeled, private single-story upstairs condo offers 2 bedrooms, 2 full bathrooms, and 951 square feet of brigh..."
3,Homes in San Diego near the beach with ocean views,1,851 53 Sapphire,San Diego,3195000,6,6.0,2817,"near beach, ocean view","This 6-bed, 6-bath listing in San Diego is listed at $3,195,000. Highlights include an ocean view and solar.",Newer Luxury Detached Home close to the Beach with spectacular ocean views! 851-853 Sapphire built in 2022 is a three story home with 5BR and 5BA with a 1B/1BA ADU. Well appoi...
4,Homes in San Diego near the beach with ocean views,2,728 Jamaica Court,San Diego,2999000,4,4.0,2386,ocean view,"This 4-bed, 4-bath listing in San Diego is listed at $2,999,000. Highlights include an ocean view and an accessory dwelling unit.","Beautiful and very spacious Mission Beach home with ocean views and a second unit. Ideal as a duplex or single-family home with ADU. (Two homes, 1 parcel, no HOA) Located just ..."
5,Homes in San Diego near the beach with ocean views,3,5236 Beachfront 83,San Diego,698000,4,4.0,1875,ocean view,"This 4-bed, 4-bath listing in San Diego is listed at $698,000. Highlights include an ocean view and a pool.","Ocean View! Ocean View! Ocean View! Wake up and wind down every day with stunning ocean views from the living room and every bedroom in this beautifully maintained 4-bedroom, 3..."
6,Homes with high ceilings and a kitchen island,1,1193 W 13th Street,Upland,880000,4,2.0,2142,high ceilings,"This 4-bed, 2-bath listing in Upland is listed at $880,000. Highlights include a pool and a covered patio.",This well-maintained single-level home offers a thoughtful layout designed for both comfortable everyday living & effortless entertaining. A spacious living room welcomes you w...
7,Homes with high ceilings and a kitchen island,2,176 Nightfall,San Jacinto,525000,4,3.0,3019,high ceilings,"This 4-bed, 3-bath listing in San Jacinto is listed at $525,000. Highlights include a mountain view and a fireplace.","Spacious 4-bedroom, 3-bath home offering over 3,000 sq. ft. of open living space with soaring ceilings and a versatile floor plan. The kitchen features a center island and ampl..."
8,Homes with high ceilings and a kitchen island,3,734 Pecan Way,Campbell,3500000,4,3.0,2588,high ceilings,"This 4-bed, 3-bath listing in Campbell is listed at $3,500,000. Highlights include new construction and a walk-in closet.",Under construction. Newly built single-story home featuring 4 bedrooms and 3 full bathrooms with a bright and functional layout designed for modern living. The open living and ...


## 7. Runtime Metrics Snapshot

The dashboard aggregates anonymous interactions recorded by the Streamlit application. It is operational context rather than a controlled benchmark, so this notebook does not add events to it. Set `API_WEB_METRICS_TOKEN` in the terminal before starting Jupyter when access to this admin-only endpoint is required.

---

In [9]:
metrics_token = os.getenv("API_WEB_METRICS_TOKEN", "")
metrics_headers = {"X-Web-Metrics-Token": metrics_token} if metrics_token else {}
metrics_response = session.get(
    f"{API_BASE_URL}/web/metrics",
    headers=metrics_headers,
    timeout=REQUEST_TIMEOUT_SECONDS,
)

if metrics_response.ok:
    runtime_metrics = metrics_response.json()
    display(pd.DataFrame([
        {
            "searches": runtime_metrics["query_volume"],
            "sessions": runtime_metrics["unique_sessions"],
            "profile_comparisons": runtime_metrics["comparison_searches"],
            "zero_result_rate": runtime_metrics["zero_result_rate"],
            "feedback_responses": runtime_metrics["satisfaction"]["responses"],
            "helpful_rate": runtime_metrics["satisfaction"]["helpful_rate"],
        }
    ]))
    pd.DataFrame([
        {
            "profile": profile,
            "searches": runtime_metrics["profile_usage"][profile],
            "api_p50_ms": runtime_metrics["profile_latency_ms"][profile]["api"]["p50"],
            "api_p90_ms": runtime_metrics["profile_latency_ms"][profile]["api"]["p90"],
            "api_p95_ms": runtime_metrics["profile_latency_ms"][profile]["api"]["p95"],
        }
        for profile in PROFILES
    ])
else:
    print(f"Runtime metrics unavailable: HTTP {metrics_response.status_code}.")

,searches,sessions,profile_comparisons,zero_result_rate,feedback_responses,helpful_rate
0,16,2,0,0.0,0,None


## 8. Quality Rerank Queue

The deployed service permits one Cross Encoder execution at a time. These distinct cache-miss probes are sent together to measure queueing directly. Results are ordered by completion time, not input order.

---

In [10]:
from concurrent.futures import ThreadPoolExecutor


quality_probes = [
    "three bedroom home in Irvine with a fireplace",
    "three bedroom home in Irvine with a fireplace please",
    "three bedroom home in Irvine with a fireplace today",
]


def quality_probe(query):
    started_at = time.perf_counter()
    response = requests.post(
        f"{API_BASE_URL}/search",
        json={"query": query, "top_k": TOP_K, "search_profile": "quality"},
        timeout=REQUEST_TIMEOUT_SECONDS,
        headers=REQUEST_HEADERS,
    )
    elapsed_ms = round((time.perf_counter() - started_at) * 1000, 2)
    body = response.json()
    return {
        "query": query,
        "http_status": response.status_code,
        "cache_status": response.headers.get("X-Cache", "BYPASS"),
        "observed_latency_ms": elapsed_ms,
        "completed_after_start_ms": round((time.perf_counter() - queue_started_at) * 1000, 2),
        "reranker_used": body.get("meta", {}).get("reranker_used"),
        "error_code": body.get("error", {}).get("code"),
    }


queue_started_at = time.perf_counter()

with ThreadPoolExecutor(max_workers=len(quality_probes)) as executor:
    queue_results = list(executor.map(quality_probe, quality_probes))

queue_results = pd.DataFrame(queue_results)
assert (queue_results["http_status"] == 200).all()
assert queue_results["reranker_used"].all()
queue_results.sort_values("completed_after_start_ms").reset_index(drop=True)

,query,http_status,cache_status,observed_latency_ms,completed_after_start_ms,reranker_used,error_code
0,three bedroom home in Irvine with a fireplace today,200,MISS,4778.47,4780.94,True,None
1,three bedroom home in Irvine with a fireplace,200,MISS,9212.49,9213.18,True,None
2,three bedroom home in Irvine with a fireplace please,200,MISS,13570.91,13572.07,True,None


## 9. Review Notes

- The held-out table is the product relevance result; the broader 40-query run is used for latency only.
- Each query/profile pair is issued twice in immediate succession, so the cache comparison reflects the same payload within its TTL.
- Quality cache misses include Cross Encoder timing; queue probes intentionally submit concurrent cache misses and should be interpreted separately.
- The details check covers the same batch endpoint used by the product page, including original listing descriptions.
- Runtime metrics reflect local web usage and should not be compared directly with the controlled benchmark.